# MNIST CNN + MLP (PyTorch)

Bản PyTorch của `Yêu cầu/examples/2. CNN + MLP/1_MNIST_CNN.ipynb`, giữ nguyên thứ tự mục và cấu hình của gốc.

- Mô hình: 3 khối [Conv 3×3 same + relu → MaxPool 2×2 → Dropout 0.2] với 64, 128, 256 filter, rồi FC 512 → FC 512 → FC 10 (dropout 0.2 sau mỗi FC ẩn).
- Huấn luyện: `categorical_crossentropy`, SGD lr 0.01 (không momentum), batch 128, 30 epoch; validation là tập test như gốc.
- Phần phụ: PCA 2 chiều của ảnh gốc, của đặc trưng `fc2` (512 chiều) và của `GlobalAveragePooling2D` sau `pool3` (256 chiều).

| | Keras gốc (tf.keras 2.14, Colab T4) | PyTorch |
|---|---|---|
| Số tham số | 1.817.610 | kiểm bằng `assert` ở mục (2) |
| Test accuracy | 0.9884 | xem mục (4) |
| Test loss | 0.0339 | xem mục (4) |
| Thời gian | 3,39 phút (T4) | xem mục (3) |

# Import libraries

Keras gốc:
```python
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import SGD
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # để import pytorch/common.py
from common import *

import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# True: chỉ chạy thử (ít dữ liệu, 1 epoch) để kiểm code trên CPU; bản nộp phải là False (chạy GPU)
CHAY_THU = False

set_seed(0)
device = get_device()

In [ ]:
num_classes = 10   # 10 classes: 0, 1,..., 9
TEN = "1_MNIST_CNN"

# (1) Data

Keras gốc:
```python
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(60000, 28, 28, 1)
x_test = x_test.reshape(10000, 28, 28, 1)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)
```

Khác Keras:
- Keras để kênh ở cuối `(N, 28, 28, 1)`; PyTorch để kênh ngay sau batch `(N, 1, 28, 28)` (NCHW), nên `unsqueeze(1)` thay cho `reshape(..., 28, 28, 1)`.
- Gốc **có** chia 255 nên ở đây cũng chia.
- Nhãn giữ dạng chỉ số lớp cho `nn.CrossEntropyLoss` thay vì `to_categorical`. Ô dưới chỉ minh họa one-hot giống ô `to_categorical(2, num_classes=3)` của gốc.

In [ ]:
import contextlib, io
with contextlib.redirect_stderr(io.StringIO()):   # ẩn thanh tiến trình khi tải lần đầu
    mnist_train = torchvision.datasets.MNIST(DATA_DIR / "mnist", train=True, download=True)
    mnist_test = torchvision.datasets.MNIST(DATA_DIR / "mnist", train=False, download=True)

x_train = (mnist_train.data.float() / 255).unsqueeze(1)   # (60000, 1, 28, 28)
x_test = (mnist_test.data.float() / 255).unsqueeze(1)     # (10000, 1, 28, 28)
y_train = mnist_train.targets.long()
y_test = mnist_test.targets.long()

if CHAY_THU:
    x_train, y_train = x_train[:2000], y_train[:2000]
    x_test, y_test = x_test[:500], y_test[:500]

print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

In [ ]:
# one-hot như keras.utils.to_categorical(2, num_classes=3)
F.one_hot(torch.tensor(2), num_classes=3)

# (2) Declare model

Keras gốc:
```python
input_image = Input(shape=(28, 28, 1), name='Input')
x = Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu', name='conv1')(input_image)
x = MaxPooling2D(pool_size=(2, 2), name='pool1')(x)
x = Dropout(rate=0.2, name='conv_dropout1')(x)
x = Conv2D(filters=128, kernel_size=(3, 3), padding='same', activation='relu', name='conv2')(x)
x = MaxPooling2D(pool_size=(2, 2), name='pool2')(x)
x = Dropout(rate=0.2, name='conv_dropout2')(x)
x = Conv2D(filters=256, kernel_size=(3, 3), padding='same', activation='relu', name='conv3')(x)
x = MaxPooling2D(pool_size=(2, 2), name='pool3')(x)
x = Dropout(rate=0.2, name='conv_dropout3')(x)
x = Flatten(name='flatten')(x)
x = Dense(units=512, activation='relu', name='fc1')(x)
x = Dropout(rate=0.2, name='fc_dropout1')(x)
x = Dense(units=512, activation='relu', name='fc2')(x)
x = Dropout(rate=0.2, name='fc_dropout2')(x)
output_label = Dense(units=num_classes, activation='softmax', name='fc3_10ways_softmax')(x)
model = Model(inputs=input_image, outputs=output_label, name='mnist_mlp')
```

Khác Keras:
- `padding='same'` với kernel 3×3 là `padding=1`. `MaxPool2d(2)` làm tròn xuống như Keras: 28 → 14 → 7 → 3.
- `nn.Linear` cần số đầu vào: sau `pool3` là 256 × 3 × 3 = 2304. Keras `Flatten` duỗi theo thứ tự (H, W, C), PyTorch theo (C, H, W); thứ tự khác nhưng số tham số và khả năng học như nhau.
- Lớp cuối trả logits (softmax nằm trong `nn.CrossEntropyLoss`).
- `forward(x, return_features=True)` trả thêm đầu ra `pool3` và `fc2` (sau relu) để làm PCA ở cuối, thay cho `Model(inputs, get_layer(...).output)`.

In [ ]:
class MnistCNN(nn.Module):
    """Tên gốc 'mnist_mlp' (gốc giữ tên của bài MLP); các lớp trùng `name=` Keras."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)      # 'same'
        self.pool1 = nn.MaxPool2d(2)
        self.conv_dropout1 = nn.Dropout(0.2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.conv_dropout2 = nn.Dropout(0.2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.conv_dropout3 = nn.Dropout(0.2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 3 * 3, 512)
        self.fc_dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(512, 512)
        self.fc_dropout2 = nn.Dropout(0.2)
        self.fc3_10ways_softmax = nn.Linear(512, num_classes)        # trả logits

    def forward(self, x, return_features=False):                    # x: (N, 1, 28, 28)
        x = self.conv_dropout1(self.pool1(F.relu(self.conv1(x))))    # (N, 64, 14, 14)
        x = self.conv_dropout2(self.pool2(F.relu(self.conv2(x))))    # (N, 128, 7, 7)
        pool3 = self.pool3(F.relu(self.conv3(x)))                    # (N, 256, 3, 3)
        x = self.flatten(self.conv_dropout3(pool3))                  # (N, 2304)
        x = self.fc_dropout1(F.relu(self.fc1(x)))                    # (N, 512)
        fc2 = F.relu(self.fc2(x))                                    # (N, 512) = đầu ra 'fc2' Keras
        logits = self.fc3_10ways_softmax(self.fc_dropout2(fc2))      # (N, 10)
        if return_features:
            return {"pool3": pool3, "fc2": fc2}, logits
        return logits


model = init_keras_like(MnistCNN(num_classes)).to(device)
summary(model, input_size=(1, 1, 28, 28))

In [ ]:
# Số tham số phải trùng "Total params: 1817610" của model.summary() Keras
n_params = count_params(model)
print("Số tham số PyTorch:", n_params)
assert n_params == 1_817_610

# (3) Train defined model

Keras gốc:
```python
loss = 'categorical_crossentropy'
lr = 0.01
model.compile(loss=loss, optimizer=SGD(lr=lr), metrics=['accuracy'])
batch_size = 128
epochs = 30
history = model.fit(x_train, y_train, validation_data=(x_test, y_test),
                    batch_size=batch_size, epochs=epochs)
```

In [ ]:
loss_fn = nn.CrossEntropyLoss()
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr)   # không momentum như SGD Keras

batch_size = 128
epochs = 1 if CHAY_THU else 30

train_loader = make_loader(x_train, y_train, batch_size=batch_size, shuffle=True)
test_loader = make_loader(x_test, y_test, batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, loss_fn, optimizer, epochs,
              metric=accuracy, metric_name="accuracy")
train_time = time.time() - starting_time
print('> training time is %.4f minutes' % (train_time / 60))

### Training time
- Gốc: GTX 1080 khoảng 4,22 phút, Core i7 khoảng 105,5 phút cho 30 epoch; lần chạy lưu trong notebook gốc (Colab T4) là 3,39 phút.
- Bản PyTorch: xem dòng `training time` ở trên.

In [ ]:
fig = plot_history(history, None if CHAY_THU else TEN, metric_name="accuracy",
                   title="MNIST CNN (PyTorch)")
plt.show()

# (4) Evaluate trained model

Kết quả gốc: `Test loss: 0.0339`, `Test accuracy: 0.9884`.

In [ ]:
score = evaluate(model, test_loader, loss_fn, accuracy)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
if not CHAY_THU:
    save_history(history, TEN, metric_name="accuracy")
    save_metrics(TEN, test_loss=score[0], test_metric=score[1], n_params=n_params,
                 epochs_ran=len(history["loss"]), best_epoch=None, train_time_s=train_time,
                 device=device_name(device), seed=0)
    ghi_so_sanh(TEN, "2. CNN + MLP", "test_acc", keras=0.9884, pytorch=score[1],
                keras_params=1817610, pytorch_params=n_params, epochs_keras=30, epochs_pytorch=epochs,
                optimizer="SGD", lr=lr, batch=batch_size, device=device_name(device),
                train_time_s=train_time, seed=0,
                ghi_chu="Keras gốc chạy trên TensorFlow 2.14, Colab T4 (3,39 phút)")

## PCA của ảnh gốc

Keras gốc dùng numpy + scikit-learn, không liên quan Keras, nên giữ nguyên: duỗi mỗi ảnh thành vector 784 chiều, PCA về 2 chiều, tô màu theo nhãn.

In [ ]:
print(tuple(x_train.shape), tuple(y_train.shape))
data_org = x_train.reshape(x_train.shape[0], -1).numpy()
data_lbl = y_train.numpy()            # gốc: np.argmax(y_train, axis=1)
print(data_org.shape, data_lbl.shape)

In [ ]:
def ve_pca(z, title, ten_file):
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=z[:, 0], y=z[:, 1], hue=data_lbl, palette='viridis', s=8, linewidth=0)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title(title, {'fontsize': 15})
    if not CHAY_THU:
        plt.savefig(results_dir(TEN) / ten_file, dpi=120, bbox_inches="tight")
    plt.show()


pca = PCA(n_components=2)
org_pca = pca.fit_transform(data_org)
print(org_pca.shape)
ve_pca(org_pca, 'Visualization of MNIST dataset using PCA (ảnh gốc, 784 chiều)', 'pca_anh_goc.png')

## Dự đoán một ảnh test ngẫu nhiên

Ô `model.get_layer()` của gốc gọi thiếu tham số (báo lỗi), nên bỏ; cách lấy lớp giữa đúng nằm ở mục dưới.

In [ ]:
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]                 # (1, 28, 28)
test_label = int(y_test[random_test_index])

pred_label = predict(model, test_img.unsqueeze(0))   # (1, 1, 28, 28)
pred_label = int(pred_label.argmax(dim=1))

plt.imshow(test_img[0], cmap='gray')                 # gốc: test_img[:, :, 0]
plt.title('true label = %d, predicted label = %d' % (test_label, pred_label))
plt.show()

## PCA của đặc trưng `fc2` và của GlobalAveragePooling sau `pool3`

Keras gốc:
```python
model_fc2 = Model(inputs=model.input, outputs=model.get_layer("fc2").output)
y_feat = model_fc2.predict(x_train)                       # (60000, 512)

x1 = model.get_layer("pool3").output
x1 = GlobalAveragePooling2D()(x1)
model2 = Model(inputs=model.input, outputs=x1)
y2_feat = model2.predict(x_train)                         # (60000, 256)
```
PyTorch: một lượt qua tập train với `return_features=True` lấy cả hai. `GlobalAveragePooling2D` = trung bình theo hai chiều không gian `mean(dim=(2, 3))`. Chạy ở `model.eval()` + `no_grad` nên dropout tắt, như `predict` của Keras.

In [ ]:
model.eval()
feat_fc2, feat_gap = [], []
with torch.no_grad():
    for xb, _ in make_loader(x_train, y_train, batch_size=512):
        feats, _ = model(xb.to(device), return_features=True)
        feat_fc2.append(feats["fc2"].cpu())
        feat_gap.append(feats["pool3"].mean(dim=(2, 3)).cpu())   # GlobalAveragePooling2D
y_feat = torch.cat(feat_fc2).numpy()
y2_feat = torch.cat(feat_gap).numpy()
print(y_feat.shape, y2_feat.shape)

In [ ]:
pca_fc2 = PCA(n_components=2)
fc2_pca = pca_fc2.fit_transform(y_feat)
print(fc2_pca.shape)
ve_pca(fc2_pca, 'Visualization of MNIST dataset using PCA (fc2, 512 chiều)', 'pca_fc2.png')

In [ ]:
pca_y2 = PCA(n_components=2)
y2_pca = pca_y2.fit_transform(y2_feat)
print(y2_pca.shape)
ve_pca(y2_pca, 'Visualization of MNIST dataset using PCA (GAP sau pool3, 256 chiều)', 'pca_gap_pool3.png')

**Nhận xét:** ở PCA của ảnh gốc các lớp chồng lên nhau nhiều; sang đặc trưng `fc2` các chữ số tách thành cụm rõ hơn, vì mạng đã học biểu diễn để phân lớp. Đặc trưng GAP sau `pool3` nằm giữa hai mức trên: đã qua các lớp tích chập nhưng chưa qua các lớp FC.